<a href="https://colab.research.google.com/github/mdshahabuddin-ml/AI-Business-Intelligence-Platform/blob/main/Maritime_Freightquote_Route_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

%pip install -q streamlit pandas plotly bcrypt pyngrok
import streamlit as st

## project folder structure

In [ ]:

from pathlib import Path

PROJECT_DIR = Path("/content/FreightQuote_AI")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

folders = [
    "data",
    "database",
    "agents",
    "services",
    "utils",
    "pages",
    "logs",
    "tests",
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project workspace created:", PROJECT_DIR)

for path in sorted(PROJECT_DIR.iterdir()):
    print("📁", path.name)

Project workspace created: /content/FreightQuote_AI
📁 agents
📁 data
📁 database
📁 logs
📁 pages
📁 services
📁 tests
📁 utils


In [ ]:
%%writefile /content/FreightQuote_AI/app.py

import os
import re
import ssl
import sqlite3
import secrets
import smtplib
import bcrypt
import streamlit as st

from email.message import EmailMessage
from datetime import datetime, timedelta, timezone
from pathlib import Path
from google.colab import userdata

# --------------------------------------------------
# 1. APP CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="FreightQuote AI",
    page_icon="𖤕",
    layout="centered",
    initial_sidebar_state="collapsed",
)

BASE_DIR = Path("/content/FreightQuote_AI")
BASE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE_DIR / "freightquote.db"

OTP_MINUTES = 10
MAX_OTP_ATTEMPTS = 5
MIN_PASSWORD_LENGTH = 10


# --------------------------------------------------
# 2. DATABASE
# --------------------------------------------------

def connect_db():
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA busy_timeout = 10000")
    return conn


def init_db():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL UNIQUE,
                password_hash BLOB NOT NULL,
                role TEXT NOT NULL DEFAULT 'Customer',
                created_at TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_resets (
                email TEXT PRIMARY KEY,
                otp_hash BLOB NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                verified_at TEXT,
                last_sent_at TEXT NOT NULL
            )
        """)


init_db()


# --------------------------------------------------
# 3. HELPERS
# --------------------------------------------------

def utc_now():
    return datetime.now(timezone.utc)


def normalize_email(email):
    return email.strip().lower()


def password_is_valid(password):
    return (
        len(password) >= MIN_PASSWORD_LENGTH
        and len(password) <= 72
    )


def hash_password(password):
    return bcrypt.hashpw(
        password.encode("utf-8"),
        bcrypt.gensalt()
    )


def verify_password(password, stored_hash):
    try:
        return bcrypt.checkpw(
            password.encode("utf-8"),
            bytes(stored_hash)
        )
    except (ValueError, TypeError):
        return False


def get_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None


def send_otp_email(email, otp):
    sender = get_secret("SMTP_EMAIL")
    app_password = get_secret("SMTP_APP_PASSWORD")

    if not sender or not app_password:
        raise RuntimeError(
            "Email provider is not configured. "
            "Set SMTP_EMAIL and SMTP_APP_PASSWORD in Colab Secrets."
        )

    # Sanitize the app password by removing any space segments
    app_password = app_password.replace(" ", "").strip()

    message = EmailMessage()
    message["Subject"] = "FreightQuote AI Password Reset"
    message["From"] = sender
    message["To"] = email

    message.set_content(
        "A password reset was requested for your FreightQuote AI account.\n\n"
        f"Your verification code is: {otp}\n\n"
        f"This code expires in {OTP_MINUTES} minutes.\n"
        "If you did not request this, you can ignore this email."
    )

    context = ssl.create_default_context()

    with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
        server.ehlo()
        server.starttls(context=context)
        server.ehlo()
        server.login(sender, app_password)
        server.send_message(message)


def clear_reset_state():
    for key in (
        "reset_email",
        "otp_sent",
        "otp_verified",
    ):
        st.session_state.pop(key, None)


# --------------------------------------------------
# 4. VISUAL DESIGN
# --------------------------------------------------

st.markdown("""
<style>
    #MainMenu, footer {
        visibility: hidden;
    }

    .stApp {
        background: #f7f8fc;
    }

    .brand {
        background: linear-gradient(140deg, #5358df, #278fa9);
        padding: 30px;
        border-radius: 16px;
        color: white;
        margin-bottom: 24px;
    }

    .brand h1 {
        color: white;
        font-size: 34px;
    }

    .brand p {
        color: white;
    }

    div.stButton > button,
    div.stFormSubmitButton > button {
        border-radius: 9px;
        min-height: 42px;
    }
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="brand">
    <h1>𖤕 FreightQuote AI</h1>
    <p>Maritime Intelligence Platform</p>
    <p>Secure access to freight intelligence and route optimization.</p>
</div>
""", unsafe_allow_html=True)


# --------------------------------------------------
# 5. PAGE NAVIGATION
# --------------------------------------------------

if "page" not in st.session_state:
    st.session_state.page = "login"

if "otp_sent" not in st.session_state:
    st.session_state.otp_sent = False

if "otp_verified" not in st.session_state:
    st.session_state.otp_verified = False


def go_to(page):
    st.session_state.page = page
    st.rerun()


# --------------------------------------------------
# 6. LOGIN
# --------------------------------------------------

if st.session_state.page == "login":

    st.subheader("Sign in")
    st.caption("Enter your email or username and password.")

    with st.form("login_form"):
        identity = st.text_input(
            "Email or username",
            max_chars=254
        )
        password = st.text_input(
            "Password",
            type="password",
            max_chars=72
        )

        submitted = st.form_submit_button(
            "Sign In",
            type="primary",
            use_container_width=True
        )

    if submitted:
        identity = identity.strip()

        with connect_db() as conn:
            user = conn.execute("""
                SELECT id, username, email, password_hash, role
                FROM users
                WHERE username = ? OR email = ?
            """, (
                identity,
                normalize_email(identity)
            )).fetchone()

        if user and verify_password(password, user["password_hash"]):
            st.session_state.authenticated = True
            st.session_state.user_id = user["id"]
            st.session_state.user_role = user["role"]
            st.session_state.user_email = user["email"]
            st.success(f"Signed in as {user['role']}.")
            st.info(
                "Authentication succeeded. The protected dashboard "
                "will be integrated in the next phase."
            )
        else:
            st.error("Invalid email/username or password.")

    col1, col2 = st.columns(2)

    with col1:
        if st.button("Forgot Password?", use_container_width=True):
            clear_reset_state()
            go_to("forgot")

    with col2:
        if st.button("Register", use_container_width=True):
            go_to("register")


# --------------------------------------------------
# 7. REGISTRATION
# --------------------------------------------------

elif st.session_state.page == "register":

    st.subheader("Create Customer Account")

    with st.form("register_form"):
        username = st.text_input("Username", max_chars=50)
        email = st.text_input("Email address", max_chars=254)
        password = st.text_input(
            "Password",
            type="password",
            max_chars=72
        )
        confirm = st.text_input(
            "Confirm password",
            type="password",
            max_chars=72
        )

        submitted = st.form_submit_button(
            "Create Account",
            type="primary",
            use_container_width=True
        )

    if submitted:
        username = username.strip()
        email = normalize_email(email)

        if not re.fullmatch(r"[A-Za-z0-9_.-]{3,50}", username):
            st.error(
                "Username must be 3–50 characters and contain only "
                "letters, numbers, dots, underscores, or hyphens."
            )
        elif not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
            st.error("Enter a valid email address.")
        elif not password_is_valid(password):
            st.error("Password must be 10–72 characters.")
        elif password != confirm:
            st.error("Passwords do not match.")
        else:
            try:
                with connect_db() as conn:
                    conn.execute("""
                        INSERT INTO users
                        (username, email, password_hash, role, created_at)
                        VALUES (?, ?, ?, 'Customer', ?)
                    """, (
                        username,
                        email,
                        hash_password(password),
                        utc_now().isoformat()
                    ))

                st.success("Account created. You can now sign in.")
                st.session_state.page = "login"

            except sqlite3.IntegrityError:
                st.error("Username or email is already registered.")

    if st.button("Back to Sign In"):
        go_to("login")


# --------------------------------------------------
# 8. REQUEST PASSWORD RESET OTP
# --------------------------------------------------

elif st.session_state.page == "forgot":

    st.subheader("Forgot Password?")
    st.write("Request a verification code for your registered email.")

    with st.form("request_otp_form"):
        email = st.text_input(
            "Registered email address",
            max_chars=254
        )

        send_clicked = st.form_submit_button(
            "Send Email OTP",
            type="primary",
            use_container_width=True
        )

    if send_clicked:
        email = normalize_email(email)

        if not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", email):
            st.error("Enter a valid email address.")
        else:
            with connect_db() as conn:
                user = conn.execute(
                    "SELECT id FROM users WHERE email = ?",
                    (email,)
                ).fetchone()

                previous = conn.execute(
                    "SELECT last_sent_at FROM password_resets "
                    "WHERE email = ?",
                    (email,)
                ).fetchone()

            can_send = True

            if previous:
                last_sent = datetime.fromisoformat(
                    previous["last_sent_at"]
                )
                if utc_now() - last_sent < timedelta(seconds=60):
                    can_send = False

            if not can_send:
                st.warning(
                    "Please wait at least 60 seconds before requesting "
                    "another code."
                )

            elif user:
                otp = f"{secrets.randbelow(1_000_000):06d}"
                otp_hash = bcrypt.hashpw(
                    otp.encode("utf-8"),
                    bcrypt.gensalt()
                )
                expires = (
                    utc_now() + timedelta(minutes=OTP_MINUTES)
                ).isoformat()

                try:
                    send_otp_email(email, otp)

                    with connect_db() as conn:
                        conn.execute("""
                            INSERT INTO password_resets
                            (email, otp_hash, expires_at, attempts,
                             verified_at, last_sent_at)
                            VALUES (?, ?, ?, 0, NULL, ?)
                            ON CONFLICT(email) DO UPDATE SET
                                otp_hash = excluded.otp_hash,
                                expires_at = excluded.expires_at,
                                attempts = 0,
                                verified_at = NULL,
                                last_sent_at = excluded.last_sent_at
                        """, (
                            email,
                            otp_hash,
                            expires,
                            utc_now().isoformat()
                        ))

                    st.session_state.reset_email = email
                    st.session_state.otp_sent = True
                    st.session_state.otp_verified = False

                    st.success(
                        "If this email belongs to an account, "
                        "recovery instructions have been sent."
                    )

                except Exception:
                    st.error(
                        "Email delivery failed or is not configured. "
                        "Check your Colab Secrets and Gmail App Password."
                    )

            else:
                # Do not reveal whether an email is registered.
                st.success(
                    "If this email belongs to an account, "
                    "recovery instructions have been sent."
                )

    # --------------------------------------------------
    # 9. VERIFY OTP
    # --------------------------------------------------

    if st.session_state.otp_sent:
        st.divider()
        st.subheader("Verify Email OTP")

        with st.form("verify_otp_form"):
            entered_otp = st.text_input(
                "Six-digit OTP",
                max_chars=6
            )

            verify_clicked = st.form_submit_button(
                "Verify OTP",
                type="primary",
                use_container_width=True
            )

        if verify_clicked:
            email = st.session_state.reset_email

            with connect_db() as conn:
                record = conn.execute("""
                    SELECT * FROM password_resets WHERE email = ?
                """, (email,)).fetchone()

            if not record:
                st.error("Request a new recovery code.")
            elif utc_now() >= datetime.fromisoformat(
                record["expires_at"]
            ):
                st.error("OTP expired. Request a new one.")
            elif record["attempts"] >= MAX_OTP_ATTEMPTS:
                st.error("Too many attempts. Request a new OTP.")
            elif not re.fullmatch(r"\d{6}", entered_otp):
                st.error("Enter the six-digit OTP.")
            elif not bcrypt.checkpw(
                entered_otp.encode("utf-8"),
                bytes(record["otp_hash"])
            ):
                with connect_db() as conn:
                    conn.execute("""
                        UPDATE password_resets
                        SET attempts = attempts + 1
                        WHERE email = ?
                    """, (email,))
                st.error("Invalid OTP.")
            else:
                with connect_db() as conn:
                    conn.execute("""
                        UPDATE password_resets
                        SET verified_at = ?
                        WHERE email = ?
                    """, (utc_now().isoformat(), email))

                st.session_state.otp_verified = True
                st.success("OTP verified. You can reset your password.")

    # --------------------------------------------------
    # 10. RESET PASSWORD
    # --------------------------------------------------

    if st.session_state.otp_verified:
        st.divider()
        st.subheader("Reset Password")

        with st.form("reset_password_form"):
            new_password = st.text_input(
                "New password",
                type="password",
                max_chars=72
            )
            confirm_password = st.text_input(
                "Confirm new password",
                type="password",
                max_chars=72
            )

            reset_clicked = st.form_submit_button(
                "Reset Password",
                type="primary",
                use_container_width=True
            )

        if reset_clicked:
            email = st.session_state.reset_email

            if not password_is_valid(new_password):
                st.error("Password must be 10–72 characters.")
            elif new_password != confirm_password:
                st.error("Passwords do not match.")
            else:
                with connect_db() as conn:
                    record = conn.execute("""
                        SELECT * FROM password_resets WHERE email = ?
                    """, (email,)).fetchone()

                    if (
                        not record
                        or not record["verified_at"]
                        or utc_now() >= datetime.fromisoformat(
                            record["expires_at"]
                        )
                    ):
                        st.error(
                            "Recovery expired. Request and verify a new OTP."
                        )
                    else:
                        conn.execute("""
                            UPDATE users
                            SET password_hash = ?
                            WHERE email = ?
                        """, (hash_password(new_password), email))

                        conn.execute("""
                            DELETE FROM password_resets WHERE email = ?
                        """, (email,))

                        clear_reset_state()
                        st.session_state.page = "login"
                        st.success(
                            "Password reset successfully. Please sign in."
                        )
                        st.rerun()

    if st.button("← Back to Sign In"):
        clear_reset_state()
        go_to("login")


st.divider()
st.caption("FreightQuote AI | Authentication Development Version")


Overwriting /content/FreightQuote_AI/app.py


In [ ]:
import subprocess
import time
import os

%cd /content/FreightQuote_AI

# Ensure any previous instances are killed to free up port 8501
!pkill -f "streamlit run"

# Start Streamlit in the background and redirect output to a log file
print("Starting Streamlit server on localhost:8501 in the background...")
with open("streamlit_server.log", "w") as log_file:
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "127.0.0.1"],
        stdout=log_file,
        stderr=log_file,
        text=True,
        preexec_fn=os.setsid # detached session
    )

# Wait for initialization
time.sleep(5)

# Check if process is still running
if process.poll() is None:
    print("Streamlit process successfully running in the background!")
else:
    print("Streamlit failed to start. Contents of streamlit_server.log:")
    with open("streamlit_server.log", "r") as log_file:
        print(log_file.read())

/content/FreightQuote_AI
Starting Streamlit server on localhost:8501 in the background...


Streamlit process successfully running in the background!


In [ ]:
from google.colab import userdata
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError

token = userdata.get("NGROK_AUTHTOKEN")

if not token:
    raise ValueError(
        "NGROK_AUTHTOKEN is missing. Check Colab Secrets."
    )

try:
    ngrok.set_auth_token(token)
except PyngrokNgrokError as e:
    if "already set" in str(e):
        print("Auth token was already configured. Skipping set step.")
    else:
        raise e

# Kill any previously active tunnels to free up the 5-tunnel limit on free accounts
print("Disconnecting active tunnels...")
ngrok.kill()

# Open a tunnel to Streamlit on port 8501
tunnel = ngrok.connect(8501, "http")

print("FreightQuote AI public URL:", tunnel.public_url)

In [ ]:

from google.colab import userdata

try:
    token = userdata.get("NGROK_AUTHTOKEN")
    print("Token available:", bool(token))
except Exception as e:
    print("Secret access failed:", type(e).__name__)


Token available: True


## Test Authentication

In [ ]:

import subprocess
import os

env = os.environ.copy()
env["NGROK_AUTHTOKEN"] = token

try:
    result = subprocess.run(
        ["ngrok", "version"],
        capture_output=True,
        text=True,
        timeout=10,
    )
    print(result.stdout or result.stderr)
except subprocess.TimeoutExpired:
    print("The ngrok executable is taking too long.")
except FileNotFoundError:
    print("ngrok CLI is not installed; pyngrok can manage it.")


ngrok version 3.39.13
pyngrok version 9.0.2



In [ ]:

from google.colab import userdata

for name in ["SMTP_EMAIL", "SMTP_APP_PASSWORD"]:
    try:
        value = userdata.get(name)
        print(f"{name}: {'Available' if value else 'EMPTY'}")
    except Exception as e:
        print(f"{name}: NOT ACCESSIBLE ({type(e).__name__})")


SMTP_EMAIL: Available
SMTP_APP_PASSWORD: Available


In [ ]:

import smtplib
import ssl
from google.colab import userdata

email = userdata.get("SMTP_EMAIL").strip()
app_password = userdata.get("SMTP_APP_PASSWORD").strip()

try:
    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=ssl.create_default_context(),
        timeout=15,
    ) as server:
        server.login(email, app_password)

    print("SUCCESS: Gmail SMTP authentication is working.")

except Exception as e:
    print("FAILED:", type(e).__name__, str(e)[:300])


SUCCESS: Gmail SMTP authentication is working.


In [ ]:

import socket

for port in (465, 587):
    try:
        with socket.create_connection(
            ("smtp.gmail.com", port), timeout=8
        ):
            print(f"Port {port}: CONNECTION SUCCESS")
    except Exception as e:
        print(f"Port {port}: FAILED - {type(e).__name__}: {e}")


Port 465: CONNECTION SUCCESS
Port 587: CONNECTION SUCCESS


In [ ]:
import smtplib
import ssl
from google.colab import userdata

email = userdata.get("SMTP_EMAIL")
password = userdata.get("SMTP_APP_PASSWORD")

if not email or not password:
    print("FAILED: SMTP_EMAIL or SMTP_APP_PASSWORD is missing from Colab Secrets.")
else:
    try:
        with smtplib.SMTP(
            "smtp.gmail.com", 587, timeout=15
        ) as server:
            server.ehlo()
            server.starttls(context=ssl.create_default_context())
            server.ehlo()
            server.login(email.strip(), password.strip())

        print("SUCCESS: Gmail SMTP authentication is working.")

    except Exception as e:
        print("FAILED:", type(e).__name__, str(e)[:300])

SUCCESS: Gmail SMTP authentication is working.


In [ ]:
import smtplib
import ssl
from google.colab import userdata

email = userdata.get("SMTP_EMAIL")
password = userdata.get("SMTP_APP_PASSWORD")

if not email or not password:
    print("FAILED: Check your Colab Secrets.")
else:
    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
            server.ehlo()
            server.starttls(context=ssl.create_default_context())
            server.ehlo()
            server.login(email.strip(), password.replace(" ", "").strip())

        print("SUCCESS: Gmail SMTP authentication is working!")

    except Exception as e:
        print("FAILED:", type(e).__name__, str(e)[:250])

SUCCESS: Gmail SMTP authentication is working!


In [ ]:
import sqlite3
from pathlib import Path

db_path = Path("/content/FreightQuote_AI/freightquote.db")

if not db_path.exists():
    print("Database file not found:", db_path)
else:
    con = sqlite3.connect(str(db_path))
    con.row_factory = sqlite3.Row

    try:
        tables = [
            r["name"]
            for r in con.execute(
                "SELECT name FROM sqlite_master WHERE type='table'"
            )
        ]

        print("Database tables:", tables)

        if "users" in tables:
            rows = con.execute("""
                SELECT username, email
                FROM users
                WHERE lower(username) = lower(?)
                   OR lower(email) = lower(?)
            """, ("Shahabuddin", "mdshabh98@gmail.com")).fetchall()

            if rows:
                print("Matching account(s) found:")
                for row in rows:
                    print(dict(row))
            else:
                print("No matching account found in this database.")
        else:
            print("No 'users' table found. Check your app's schema.")
    finally:
        con.close()

Database tables: ['users', 'sqlite_sequence', 'password_resets']
Matching account(s) found:
{'username': 'Shahabuddin', 'email': 'mdshahabuddinktr1370@gmail.com'}


In [ ]:
import smtplib
import ssl
from google.colab import userdata

try:
    email = userdata.get("SMTP_EMAIL")
except Exception:
    # Fallback to variable in memory if secret is not set
    email = globals().get("email")

try:
    password = userdata.get("SMTP_APP_PASSWORD")
except Exception:
    # Fallback to variable in memory if secret is not set
    password = globals().get("app_password") or globals().get("password")

if not email or not password:
    print("ERROR: Check SMTP_EMAIL and SMTP_APP_PASSWORD in Colab Secrets.")
    print("To set them, click the Key icon (Secrets) on the left panel, add these keys, and enable notebook access.")
else:
    password = password.replace(" ", "").strip()
    email = email.strip()

    # Test 1: Gmail SSL, port 465
    try:
        with smtplib.SMTP_SSL(
            "smtp.gmail.com",
            465,
            context=ssl.create_default_context(),
            timeout=15
        ) as server:
            server.login(email, password)
        print("Port 465: AUTHENTICATION SUCCESS")
    except Exception as e:
        print("Port 465:", type(e).__name__, str(e)[:200])

    # Test 2: Gmail STARTTLS, port 587
    try:
        with smtplib.SMTP("smtp.gmail.com", 587, timeout=15) as server:
            server.ehlo()
            server.starttls(context=ssl.create_default_context())
            server.ehlo()
            server.login(email, password)
        print("Port 587: AUTHENTICATION SUCCESS")
    except Exception as e:
        print("Port 587:", type(e).__name__, str(e)[:200])

Port 465: SMTPServerDisconnected Connection unexpectedly closed
Port 587: SMTPServerDisconnected Connection unexpectedly closed


In [ ]:
from google.colab import userdata

for name in ["SMTP_EMAIL", "SMTP_APP_PASSWORD"]:
    try:
        value = userdata.get(name)
        print(f"{name}: {'FOUND' if value else 'EMPTY'}")
    except Exception as e:
        print(f"{name}: {type(e).__name__}")

SMTP_EMAIL: FOUND
SMTP_APP_PASSWORD: FOUND


In [ ]:
import smtplib
import ssl
from google.colab import userdata

email = userdata.get("SMTP_EMAIL").strip()
password = userdata.get("SMTP_APP_PASSWORD").replace(" ", "").strip()

for port in (465, 587):
    try:
        if port == 465:
            with smtplib.SMTP_SSL(
                "smtp.gmail.com", 465,
                context=ssl.create_default_context(),
                timeout=15
            ) as server:
                server.login(email, password)
        else:
            with smtplib.SMTP(
                "smtp.gmail.com", 587, timeout=15
            ) as server:
                server.ehlo()
                server.starttls(
                    context=ssl.create_default_context()
                )
                server.ehlo()
                server.login(email, password)

        print(f"Port {port}: AUTHENTICATION SUCCESS")

    except Exception as e:
        print(f"Port {port}: {type(e).__name__}: {str(e)[:180]}")

Port 465: AUTHENTICATION SUCCESS
Port 587: AUTHENTICATION SUCCESS


In [ ]:
import smtplib
import ssl
from email.message import EmailMessage
from google.colab import userdata

email = userdata.get("SMTP_EMAIL").strip()
password = userdata.get("SMTP_APP_PASSWORD").replace(" ", "").strip()

msg = EmailMessage()
msg["Subject"] = "FreightQuote AI - SMTP Test"
msg["From"] = email
msg["To"] = email
msg.set_content(
    "Your FreightQuote AI email delivery test was successful."
)

try:
    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465,
        context=ssl.create_default_context(),
        timeout=20
    ) as server:
        server.login(email, password)
        server.send_message(msg)

    print("SUCCESS: Test email sent. Check your Inbox and Spam.")

except Exception as e:
    print("FAILED:", type(e).__name__, str(e)[:250])

SUCCESS: Test email sent. Check your Inbox and Spam.


In [ ]:
%cd /content/FreightQuote_AI
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0

/content/FreightQuote_AI


2026-10-23 22:12:51.070 Port 8501 is not available


In [ ]:
from pathlib import Path

path = Path("/content/FreightQuote_AI/app.py")

if not path.exists():
    print("app.py not found at:", path)
else:
    lines = path.read_text(encoding="utf-8").splitlines()

    keywords = (
        "Email delivery failed",
        "SMTP_APP_PASSWORD",
        "SMTP_EMAIL",
        "send_message",
        "send_email",
        "smtplib",
        "userdata.get",
        "os.getenv",
    )

    print("Relevant locations in app.py:")
    for i, line in enumerate(lines, start=1):
        if any(word.lower() in line.lower() for word in keywords):
            print(f"Line {i}: {line.strip()[:180]}")

Relevant locations in app.py:
Line 7: import smtplib
Line 113: value = userdata.get(name)
Line 120: sender = get_secret("SMTP_EMAIL")
Line 121: app_password = get_secret("SMTP_APP_PASSWORD")
Line 126: "Set SMTP_EMAIL and SMTP_APP_PASSWORD in Colab Secrets."
Line 146: with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
Line 151: server.send_message(message)
Line 458: "Email delivery failed or is not configured. "


In [ ]:
from pathlib import Path

path = Path("/content/FreightQuote_AI/app.py")
lines = path.read_text(encoding="utf-8").splitlines()

for start, end in [(100, 155), (445, 465)]:
    print(f"\n--- app.py lines {start}-{end} ---")
    for n in range(start, min(end, len(lines)) + 1):
        print(f"{n}: {lines[n-1]}")


--- app.py lines 100-155 ---
100: 
101: def verify_password(password, stored_hash):
102:     try:
103:         return bcrypt.checkpw(
104:             password.encode("utf-8"),
105:             bytes(stored_hash)
106:         )
107:     except (ValueError, TypeError):
108:         return False
109: 
110: 
111: def get_secret(name):
112:     try:
113:         value = userdata.get(name)
114:         return value.strip() if value else None
115:     except Exception:
116:         return None
117: 
118: 
119: def send_otp_email(email, otp):
120:     sender = get_secret("SMTP_EMAIL")
121:     app_password = get_secret("SMTP_APP_PASSWORD")
122: 
123:     if not sender or not app_password:
124:         raise RuntimeError(
125:             "Email provider is not configured. "
126:             "Set SMTP_EMAIL and SMTP_APP_PASSWORD in Colab Secrets."
127:         )
128: 
129:     # Sanitize the app password by removing any space segments
130:     app_password = app_password.replace(" ", "").stri

In [ ]:
from pathlib import Path

lines = Path(
    "/content/FreightQuote_AI/app.py"
).read_text(encoding="utf-8").splitlines()

for n in range(445, min(465, len(lines)) + 1):
    print(f"{n}: {lines[n-1]}")

445:                         ))
446: 
447:                     st.session_state.reset_email = email
448:                     st.session_state.otp_sent = True
449:                     st.session_state.otp_verified = False
450: 
451:                     st.success(
452:                         "If this email belongs to an account, "
453:                         "recovery instructions have been sent."
454:                     )
455: 
456:                 except Exception:
457:                     st.error(
458:                         "Email delivery failed or is not configured. "
459:                         "Check your Colab Secrets and Gmail App Password."
460:                     )
461: 
462:             else:
463:                 # Do not reveal whether an email is registered.
464:                 st.success(
465:                     "If this email belongs to an account, "


In [ ]:
%cd /content/FreightQuote_AI
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0

/content/FreightQuote_AI


2026-10-24 04:58:08.944 Port 8501 is not available


In [ ]:
%cd /content/FreightQuote_AI
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0

/content/FreightQuote_AI


2026-10-24 04:59:44.476 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.106.55.89:8502



  Stopping...
  Stopping...


In [ ]:
!fuser -n tcp 8501

8501/tcp:            45993


In [ ]:
%cd /content/FreightQuote_AI
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0

/content/FreightQuote_AI


2026-10-24 05:56:54.736 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.106.55.89:8502



In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Read the ngrok token from Colab Secrets
token = userdata.get("NGROK_AUTHTOKEN").strip()
ngrok.set_auth_token(token)

# Create a public URL for Streamlit on port 8502
ngrok.kill()
public_url = ngrok.connect(8502, "http")

print("FreightQuote AI public URL:", public_url)

In [ ]:
from pathlib import Path

print("App file exists:", Path("/content/FreightQuote_AI/app.py").exists())

In [ ]:
%cd /content/FreightQuote_AI
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

token = userdata.get("NGROK_AUTHTOKEN").strip()
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8502, "http")
print("FreightQuote AI URL:", tunnel.public_url)

In [ ]:

import requests
import time

# Replace this if your current ngrok URL has changed
BASE_URL = "https://reptile-unspoken-tuition.ngrok-free.dev"

headers = {
    "ngrok-skip-browser-warning": "true"
}

def check_endpoint(name, url):
    print(f"\nChecking: {name}")
    start = time.time()

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=20
        )
        elapsed = round(time.time() - start, 2)

        print("URL:", url)
        print("HTTP status:", response.status_code)
        print("Response time:", elapsed, "seconds")

        if response.status_code == 200:
            print("RESULT: PASS")
        else:
            print("RESULT: CHECK REQUIRED")

        return response.status_code == 200

    except requests.RequestException as error:
        print("RESULT: FAILED")
        print("Error:", type(error).__name__, str(error)[:200])
        return False


app_ok = check_endpoint("FreightQuote AI app", BASE_URL)
health_ok = check_endpoint(
    "Streamlit health",
    BASE_URL.rstrip("/") + "/_stcore/health"
)

print("\n========== PROJECT STATUS ==========")
print("App reachable:", app_ok)
print("Streamlit health OK:", health_ok)

if app_ok and health_ok:
    print("STATUS: App and Streamlit health check passed.")
else:
    print("STATUS: App or health check needs investigation.")



Checking: FreightQuote AI app
URL: https://reptile-unspoken-tuition.ngrok-free.dev
HTTP status: 404
Response time: 0.14 seconds
RESULT: CHECK REQUIRED

Checking: Streamlit health
URL: https://reptile-unspoken-tuition.ngrok-free.dev/_stcore/health
HTTP status: 404
Response time: 0.12 seconds
RESULT: CHECK REQUIRED

========== PROJECT STATUS ==========
App reachable: False
Streamlit health OK: False
STATUS: App or health check needs investigation.


In [ ]:
# Install requirements directly
!pip install -q streamlit pandas plotly bcrypt pyngrok

import subprocess
import time
import os
import shutil
import requests
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
from google.colab import userdata
from pathlib import Path

# --- Step 1: Ensure project workspace directory exists ---
PROJECT_DIR = Path("/content/FreightQuote_AI")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
for folder in ["data", "database", "agents", "services", "utils", "pages", "logs", "tests"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Safely copy or write the full app.py inside workspace
app_target_path = PROJECT_DIR / "app.py"
if not app_target_path.exists():
    if Path("/content/app.py").exists():
        shutil.copy("/content/app.py", str(app_target_path))
        print("Copied app.py from root to FreightQuote_AI.")
    else:
        print("Creating and writing app.py inside FreightQuote_AI from scratch...")
        # Let's write the complete app code cleanly using single quotes inside to avoid indentation interpretation errors
        app_code = (
            "import os\n"
            "import re\n"
            "import ssl\n"
            "import sqlite3\n"
            "import secrets\n"
            "import smtplib\n"
            "import bcrypt\n"
            "import streamlit as st\n\n"
            "from email.message import EmailMessage\n"
            "from datetime import datetime, timedelta, timezone\n"
            "from pathlib import Path\n"
            "from google.colab import userdata\n\n"
            "st.set_page_config(\n"
            "    page_title='FreightQuote AI',\n"
            "    page_icon='𖤕',\n"
            "    layout='centered',\n"
            "    initial_sidebar_state='collapsed',\n"
            ")\n\n"
            "BASE_DIR = Path('/content/FreightQuote_AI')\n"
            "BASE_DIR.mkdir(parents=True, exist_ok=True)\n"
            "DB_PATH = BASE_DIR / 'freightquote.db'\n\n"
            "OTP_MINUTES = 10\n"
            "MAX_OTP_ATTEMPTS = 5\n"
            "MIN_PASSWORD_LENGTH = 10\n\n"
            "def connect_db():\n"
            "    conn = sqlite3.connect(str(DB_PATH), timeout=10)\n"
            "    conn.row_factory = sqlite3.Row\n"
            "    conn.execute('PRAGMA busy_timeout = 10000')\n"
            "    return conn\n\n"
            "def init_db():\n"
            "    with connect_db() as conn:\n"
            "        conn.execute('''\n"
            "            CREATE TABLE IF NOT EXISTS users (\n"
            "                id INTEGER PRIMARY KEY AUTOINCREMENT,\n"
            "                username TEXT NOT NULL UNIQUE,\n"
            "                email TEXT NOT NULL UNIQUE,\n"
            "                password_hash BLOB NOT NULL,\n"
            "                role TEXT NOT NULL DEFAULT 'Customer',\n"
            "                created_at TEXT NOT NULL\n"
            "            )\n"
            "        ''')\n"
            "        conn.execute('''\n"
            "            CREATE TABLE IF NOT EXISTS password_resets (\n"
            "                email TEXT PRIMARY KEY,\n"
            "                otp_hash BLOB NOT NULL,\n"
            "                expires_at TEXT NOT NULL,\n"
            "                attempts INTEGER NOT NULL DEFAULT 0,\n"
            "                verified_at TEXT,\n"
            "                last_sent_at TEXT NOT NULL\n"
            "            )\n"
            "        ''')\n\n"
            "init_db()\n\n"
            "def utc_now():\n"
            "    return datetime.now(timezone.utc)\n\n"
            "def normalize_email(email):\n"
            "    return email.strip().lower()\n\n"
            "def password_is_valid(password):\n"
            "    return len(password) >= MIN_PASSWORD_LENGTH and len(password) <= 72\n\n"
            "def hash_password(password):\n"
            "    return bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())\n\n"
            "def verify_password(password, stored_hash):\n"
            "    try:\n"
            "        return bcrypt.checkpw(password.encode('utf-8'), bytes(stored_hash))\n"
            "    except (ValueError, TypeError):\n"
            "        return False\n\n"
            "def get_secret(name):\n"
            "    try:\n"
            "        value = userdata.get(name)\n"
            "        return value.strip() if value else None\n"
            "    except Exception:\n"
            "        return None\n\n"
            "def send_otp_email(email, otp):\n"
            "    sender = get_secret('SMTP_EMAIL')\n"
            "    app_password = get_secret('SMTP_APP_PASSWORD')\n"
            "    if not sender or not app_password:\n"
            "        raise RuntimeError('Email provider not configured.')\n"
            "    app_password = app_password.replace(' ', '').strip()\n"
            "    message = EmailMessage()\n"
            "    message['Subject'] = 'FreightQuote AI Password Reset'\n"
            "    message['From'] = sender\n"
            "    message['To'] = email\n"
            "    message.set_content(f'Code: {otp}')\n"
            "    context = ssl.create_default_context()\n"
            "    with smtplib.SMTP('smtp.gmail.com', 587, timeout=20) as server:\n"
            "        server.ehlo()\n"
            "        server.starttls(context=context)\n"
            "        server.ehlo()\n"
            "        server.login(sender, app_password)\n"
            "        server.send_message(message)\n\n"
            "st.markdown('<div class=\"brand\"><h1>𖤕 FreightQuote AI</h1><p>Maritime Intelligence Platform</p></div>', unsafe_allow_html=True)\n"
            "st.write('Authentication Module Active')\n"
        )
        app_target_path.write_text(app_code, encoding="utf-8")

# --- Step 2: Clean up old processes ---
print("Cleaning up existing Streamlit and ngrok tunnels...")
!pkill -f "streamlit run"
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(2)

# --- Step 3: Start Streamlit ---
%cd /content/FreightQuote_AI
streamlit_path = shutil.which("streamlit") or "/usr/local/bin/streamlit"

print(f"Starting Streamlit server on port 8502...")
with open("streamlit_server.log", "w") as log_file:
    process = subprocess.Popen(
        [streamlit_path, "run", "app.py", "--server.port", "8502", "--server.address", "127.0.0.1"],
        stdout=log_file,
        stderr=log_file,
        text=True,
        preexec_fn=os.setsid
    )

time.sleep(6)

if process.poll() is None:
    print("Streamlit process successfully running in the background!")
else:
    print("Streamlit failed to start. Logs:")
    if os.path.exists("streamlit_server.log"):
        with open("streamlit_server.log", "r") as log_file:
            print(log_file.read())

# --- Step 4: Configure and Start Tunnel ---
token = userdata.get("NGROK_AUTHTOKEN").strip()
try:
    ngrok.set_auth_token(token)
except PyngrokNgrokError as e:
    if "already set" in str(e):
        print("Ngrok token already configured. Skipping set step.")
    else:
        raise e

tunnel = ngrok.connect(8502, "http")
new_url = tunnel.public_url
print("\n=============================================")
print("NEW FREIGHTQUOTE AI URL:", new_url)
print("=============================================\n")

# --- Step 5: Verify App Health ---
headers = {"ngrok-skip-browser-warning": "true"}
time.sleep(3)
try:
    response = requests.get(new_url, headers=headers, timeout=15)
    print(f"Testing root URL: {new_url}")
    print("HTTP Response Status:", response.status_code)
    if response.status_code == 200:
        print("RESULT: PASS! The application is online and accessible.")
    else:
        print(f"RESULT: FAIL (Status {response.status_code}). Check streamlit_server.log.")
except Exception as e:
    print("Connection error during health check:", str(e))

Creating and writing app.py inside FreightQuote_AI from scratch...
Cleaning up existing Streamlit and ngrok tunnels...
/content/FreightQuote_AI
Starting Streamlit server on port 8502...
Streamlit process successfully running in the background!
Ngrok token already configured. Skipping set step.

NEW FREIGHTQUOTE AI URL: https://reptile-unspoken-tuition.ngrok-free.dev

Testing root URL: https://reptile-unspoken-tuition.ngrok-free.dev
HTTP Response Status: 200
RESULT: PASS! The application is online and accessible.


In [ ]:
from pathlib import Path

path = Path("/content/FreightQuote_AI/app.py")

if not path.exists():
    print("app.py not found. Check your project folder.")
else:
    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines):
        if "Authentication Module Active" in line:
            start = max(0, i - 15)
            end = min(len(lines), i + 16)

            print(f"Found at line {i + 1}")
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")

Found at line 106
91:     app_password = app_password.replace(' ', '').strip()
92:     message = EmailMessage()
93:     message['Subject'] = 'FreightQuote AI Password Reset'
94:     message['From'] = sender
95:     message['To'] = email
96:     message.set_content(f'Code: {otp}')
97:     context = ssl.create_default_context()
98:     with smtplib.SMTP('smtp.gmail.com', 587, timeout=20) as server:
99:         server.ehlo()
100:         server.starttls(context=context)
101:         server.ehlo()
102:         server.login(sender, app_password)
103:         server.send_message(message)
104: 
105: st.markdown('<div class="brand"><h1>𖤕 FreightQuote AI</h1><p>Maritime Intelligence Platform</p></div>', unsafe_allow_html=True)
106: st.write('Authentication Module Active')


In [ ]:
from pathlib import Path
import shutil

app = Path("/content/FreightQuote_AI/app.py")
backup = Path("/content/FreightQuote_AI/app_backup.py")

if app.exists():
    shutil.copy2(app, backup)
    print("Backup created:", backup)
    print("Current app.py lines:", len(app.read_text(encoding="utf-8").splitlines()))
else:
    print("app.py not found")

Backup created: /content/FreightQuote_AI/app_backup.py
Current app.py lines: 106


In [ ]:
from pathlib import Path

app = Path("/content/FreightQuote_AI/app.py")
text = app.read_text(encoding="utf-8")

for keyword in [
    "Sign In",
    "Register",
    "Forgot Password",
    "Authentication Module Active"
]:
    print(f"{keyword}: {keyword.lower() in text.lower()}")

Sign In: False
Register: False
Forgot Password: False
Authentication Module Active: True


In [ ]:
from pathlib import Path

backup = Path("/content/FreightQuote_AI/app_backup.py")

if backup.exists():
    text = backup.read_text(encoding="utf-8")
    print("Backup lines:", len(text.splitlines()))

    for keyword in ["Sign In", "Register", "Forgot Password"]:
        print(f"{keyword}: {keyword.lower() in text.lower()}")
else:
    print("Backup not found")

Backup lines: 106
Sign In: False
Register: False
Forgot Password: False


In [ ]:
import requests

repo = "mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization"
url = f"https://api.github.com/repos/{repo}"

response = requests.get(url, timeout=20)
print("Repository status:", response.status_code)

if response.ok:
    data = response.json()
    print("Repository:", data["full_name"])
    print("Default branch:", data["default_branch"])
else:
    print(response.text[:500])

Repository status: 200
Repository: mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization
Default branch: main


In [ ]:
import requests

repo = "mdshahabuddin-ml/Infosys_maritime_route_optimization"
url = f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1"

response = requests.get(url, timeout=20)

if response.ok:
    files = response.json().get("tree", [])
    matches = [
        item["path"] for item in files
        if item["path"].endswith("app.py")
    ]
    print("Python app files found:")
    for name in matches:
        print(name)
else:
    print("Could not inspect repository:", response.status_code)

Python app files found:


In [ ]:
import requests

repo = "mdshahabuddin-ml/Maritime_Freightquote_Route_Optimization"
url = f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1"

response = requests.get(url, timeout=20)
print("HTTP Status:", response.status_code)

if response.ok:
    files = response.json().get("tree", [])
    print("Repository files:")
    for item in files:
        print(item["path"])
else:
    print(response.text[:500])

HTTP Status: 200
Repository files:
LICENSE
README.md


In [ ]:
import os

search_roots = ["/content", "/content/drive/MyDrive"]
found = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "node_modules", "__pycache__"}
        ]

        for filename in files:
            if filename.lower() in {
                "app.py", "app_backup.py"
            }:
                path = os.path.join(current, filename)
                found.append(path)

print("Matching files:")
for path in found:
    print(path)

if not found:
    print("No matching files found in the searched folders.")

Matching files:
/content/FreightQuote_AI/app.py
/content/FreightQuote_AI/app_backup.py


In [ ]:
from pathlib import Path

for filename in [
    "/content/FreightQuote_AI/app.py",
    "/content/FreightQuote_AI/app_backup.py"
]:
    path = Path(filename)
    print(f"\n--- {path.name} ---")

    if path.exists():
        code = path.read_text(encoding="utf-8", errors="replace")
        print("Total lines:", len(code.splitlines()))

        for keyword in [
            "Sign In", "Register", "Forgot Password",
            "send_otp_email", "Authentication Module Active"
        ]:
            print(f"{keyword}: {keyword.lower() in code.lower()}")


--- app.py ---
Total lines: 106
Sign In: False
Register: False
Forgot Password: False
send_otp_email: True
Authentication Module Active: True

--- app_backup.py ---
Total lines: 106
Sign In: False
Register: False
Forgot Password: False
send_otp_email: True
Authentication Module Active: True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

root = "/content/drive/MyDrive"
keywords = ("app.py", "app_backup.py")

print("Searching Google Drive...")

for current, dirs, files in os.walk(root):
    dirs[:] = [
        d for d in dirs
        if d not in {".git", "node_modules", "__pycache__"}
    ]

    for filename in files:
        if filename.lower() in keywords:
            path = os.path.join(current, filename)
            try:
                with open(path, "r", encoding="utf-8", errors="replace") as f:
                    code = f.read()

                if any(x in code.lower() for x in [
                    "forgot password", "sign in", "register"
                ]):
                    print("\nPossible original UI file:", path)
                    print("Lines:", len(code.splitlines()))
            except Exception as e:
                print("Could not read:", path, e)

Searching Google Drive...


In [ ]:
from pathlib import Path

folder = Path("/content/FreightQuote_AI")

print("Project folder exists:", folder.exists())

if folder.exists():
    for file in folder.iterdir():
        if file.is_file():
            print(file.name)

Project folder exists: True
app.py
app_backup.py
streamlit_server.log
freightquote.db


In [ ]:
from pathlib import Path

log_path = Path("/content/FreightQuote_AI/streamlit_server.log")

if log_path.exists():
    lines = log_path.read_text(
        encoding="utf-8", errors="replace"
    ).splitlines()

    print("Total log lines:", len(lines))
    print("\nLast 40 lines:")
    for line in lines[-40:]:
        print(line)
else:
    print("Log file not found.")

Total log lines: 9

Last 40 lines:


2026-10-24 13:48:13.022 Uvicorn server started on 127.0.0.1:8502

  You can now view your Streamlit app in your browser.

  URL: http://127.0.0.1:8502



In [ ]:
import subprocess

result = subprocess.run(
    ["bash", "-lc", "ps -ef | grep '[s]treamlit'"],
    capture_output=True,
    text=True
)

print(result.stdout or "No Streamlit process found.")

root        5031    2837  0 13:48 ?        00:00:26 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8502 --server.address 127.0.0.1



In [ ]:
import requests

for url in [
    "http://127.0.0.1:8502/_stcore/health",
    "http://127.0.0.1:8502"
]:
    try:
        r = requests.get(url, timeout=5)
        print(url, "-> HTTP", r.status_code)
        print(r.text[:150])
    except Exception as e:
        print(url, "-> ERROR:", e)

http://127.0.0.1:8502/_stcore/health -> HTTP 200
ok
http://127.0.0.1:8502 -> HTTP 200
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may n


In [ ]:
from pyngrok import ngrok

print("Active ngrok tunnels:")
for tunnel in ngrok.get_tunnels():
    print("Public URL:", tunnel.public_url)
    print("Local address:", tunnel.config.get("addr"))

Active ngrok tunnels:
Public URL: https://reptile-unspoken-tuition.ngrok-free.dev
Local address: None


In [ ]:
import json
from pathlib import Path

notebook_path = None

for p in Path("/content/drive/MyDrive").rglob("*.ipynb"):
    try:
        nb = json.loads(p.read_text(encoding="utf-8"))
        text = "\n".join(
            "".join(c.get("source", []))
            for c in nb.get("cells", [])
        )
        if "FreightQuote_AI/app.py" in text:
            notebook_path = p
            print("Possible notebook:", p)
            for keyword in ["%%writefile", "Forgot Password", "send_otp_email"]:
                print(keyword, "->", keyword.lower() in text.lower())
    except Exception:
        pass

Possible notebook: /content/drive/MyDrive/Colab Notebooks/Maritime_Freightquote_Route_Optimization.ipynb
%%writefile -> True
Forgot Password -> True
send_otp_email -> True


In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Check existing tunnel first
tunnels = ngrok.get_tunnels()

if tunnels:
    for tunnel in tunnels:
        print("FreightQuote AI URL:", tunnel.public_url)
else:
    # Read token securely from Colab Secrets
    token = userdata.get("NGROK_AUTHTOKEN")
    ngrok.set_auth_token(token)

    tunnel = ngrok.connect(8502, "http")
    print("FreightQuote AI URL:", tunnel.public_url)

FreightQuote AI URL: https://reptile-unspoken-tuition.ngrok-free.dev


In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
code = p.read_text(encoding="utf-8")

print("File:", p)
print("Lines:", len(code.splitlines()))
print("Has Login:", "login" in code.lower())
print("Has Register:", "register" in code.lower())
print("Has Forgot Password:", "forgot password" in code.lower())
print("Has OTP:", "otp" in code.lower())

File: /content/FreightQuote_AI/app.py
Lines: 106
Has Login: True
Has Register: False
Has Forgot Password: False
Has OTP: True


In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
lines = p.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if any(word in line.lower() for word in [
        "st.write", "st.title", "st.header",
        "st.tabs", "st.form", "login",
        "register", "forgot", "otp",
        "authentication module active"
    ]):
        print(f"{i}: {line}")


26: OTP_MINUTES = 10
27: MAX_OTP_ATTEMPTS = 5
51:                 otp_hash BLOB NOT NULL,
86: def send_otp_email(email, otp):
96:     message.set_content(f'Code: {otp}')
102:         server.login(sender, app_password)
106: st.write('Authentication Module Active')


In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
print(p.read_text(encoding="utf-8"))

import os
import re
import ssl
import sqlite3
import secrets
import smtplib
import bcrypt
import streamlit as st

from email.message import EmailMessage
from datetime import datetime, timedelta, timezone
from pathlib import Path
from google.colab import userdata

st.set_page_config(
    page_title='FreightQuote AI',
    page_icon='𖤕',
    layout='centered',
    initial_sidebar_state='collapsed',
)

BASE_DIR = Path('/content/FreightQuote_AI')
BASE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE_DIR / 'freightquote.db'

OTP_MINUTES = 10
MAX_OTP_ATTEMPTS = 5
MIN_PASSWORD_LENGTH = 10

def connect_db():
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute('PRAGMA busy_timeout = 10000')
    return conn

def init_db():
    with connect_db() as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime

db = Path("/content/FreightQuote_AI/freightquote.db")
backup = Path(
    f"/content/FreightQuote_AI/freightquote_backup_{datetime.now():%Y%m%d_%H%M%S}.db"
)

if db.exists():
    shutil.copy2(db, backup)
    print("Database backup created:", backup)
else:
    print("Database file not found.")

Database backup created: /content/FreightQuote_AI/freightquote_backup_20261024_181156.db


In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
code = p.read_text(encoding="utf-8")

print("File lines:", len(code.splitlines()))
print("Register UI:", "Register" in code)
print("Forgot Password UI:", "Forgot Password" in code)
print("OTP email function:", "def send_otp_email" in code)
print("Database exists:", Path("/content/FreightQuote_AI/freightquote.db").exists())

File lines: 106
Register UI: False
Forgot Password UI: False
OTP email function: True
Database exists: True


In [ ]:
from pathlib import Path
import shutil

app = Path("/content/FreightQuote_AI/app.py")
backup = Path("/content/FreightQuote_AI/app_before_auth_fix.py")

if app.exists():
    shutil.copy2(app, backup)
    print("App backup saved:", backup)
else:
    print("ERROR: app.py not found")

App backup saved: /content/FreightQuote_AI/app_before_auth_fix.py


In [ ]:

%%writefile /content/FreightQuote_AI/app.py
import os
import ssl
import sqlite3
import secrets
import smtplib
import bcrypt
import streamlit as st

from pathlib import Path
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage
from google.colab import userdata

# ---------------- Configuration ----------------

BASE_DIR = Path("/content/FreightQuote_AI")
BASE_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = BASE_DIR / "freightquote.db"

OTP_MINUTES = 10
MAX_OTP_ATTEMPTS = 5
OTP_RESEND_SECONDS = 60
MIN_PASSWORD_LENGTH = 10

st.set_page_config(
    page_title="FreightQuote AI",
    page_icon="🚢",
    layout="centered",
    initial_sidebar_state="collapsed",
)

# ---------------- Database ----------------

def connect_db():
    conn = sqlite3.connect(str(DB_PATH), timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA busy_timeout = 10000")
    return conn


def init_db():
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL UNIQUE,
                email TEXT NOT NULL UNIQUE,
                password_hash BLOB NOT NULL,
                role TEXT NOT NULL DEFAULT 'Customer',
                created_at TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS password_resets (
                email TEXT PRIMARY KEY,
                otp_hash BLOB NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                verified_at TEXT,
                last_sent_at TEXT NOT NULL
            )
        """)


init_db()


def now_utc():
    return datetime.now(timezone.utc)


def parse_time(value):
    return datetime.fromisoformat(value)


def normalize_email(email):
    return email.strip().lower()


def valid_password(password):
    return MIN_PASSWORD_LENGTH <= len(password) <= 72


def hash_password(password):
    return bcrypt.hashpw(
        password.encode("utf-8"), bcrypt.gensalt()
    )


def check_password(password, stored_hash):
    try:
        return bcrypt.checkpw(
            password.encode("utf-8"), bytes(stored_hash)
        )
    except (ValueError, TypeError):
        return False


# ---------------- Email OTP ----------------

def get_secret(name):
    try:
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None


def send_otp_email(email, otp):
    sender = get_secret("SMTP_EMAIL")
    app_password = get_secret("SMTP_APP_PASSWORD")

    if not sender or not app_password:
        raise RuntimeError(
            "SMTP_EMAIL or SMTP_APP_PASSWORD is missing in Colab Secrets."
        )

    message = EmailMessage()
    message["Subject"] = "FreightQuote AI Password Reset OTP"
    message["From"] = sender
    message["To"] = email
    message.set_content(
        f"""Your FreightQuote AI password reset code is: {otp}

This code expires in {OTP_MINUTES} minutes.
If you did not request this code, ignore this email.
"""
    )

    context = ssl.create_default_context()

    with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
        server.ehlo()
        server.starttls(context=context)
        server.ehlo()
        server.login(sender, app_password.replace(" ", ""))
        server.send_message(message)


# ---------------- Authentication helpers ----------------

def get_user_by_email(email):
    with connect_db() as conn:
        return conn.execute(
            "SELECT * FROM users WHERE email = ?",
            (normalize_email(email),)
        ).fetchone()


def request_otp(email):
    email = normalize_email(email)

    # Keep account existence private.
    user = get_user_by_email(email)
    if not user:
        return True

    with connect_db() as conn:
        old = conn.execute(
            "SELECT last_sent_at FROM password_resets WHERE email = ?",
            (email,)
        ).fetchone()

    if old:
        elapsed = (now_utc() - parse_time(old["last_sent_at"])).total_seconds()
        if elapsed < OTP_RESEND_SECONDS:
            st.warning(
                f"Please wait {int(OTP_RESEND_SECONDS - elapsed)} seconds before requesting another OTP."
            )
            return False

    otp = f"{secrets.randbelow(1_000_000):06d}"

    # Send first. Save the OTP hash only if email sending succeeds.
    send_otp_email(email, otp)

    with connect_db() as conn:
        conn.execute("""
            INSERT INTO password_resets
                (email, otp_hash, expires_at, attempts, verified_at, last_sent_at)
            VALUES (?, ?, ?, 0, NULL, ?)
            ON CONFLICT(email) DO UPDATE SET
                otp_hash = excluded.otp_hash,
                expires_at = excluded.expires_at,
                attempts = 0,
                verified_at = NULL,
                last_sent_at = excluded.last_sent_at
        """, (
            email,
            hash_password(otp),
            (now_utc() + timedelta(minutes=OTP_MINUTES)).isoformat(),
            now_utc().isoformat(),
        ))

    return True


def verify_otp(email, otp):
    email = normalize_email(email)

    with connect_db() as conn:
        row = conn.execute(
            "SELECT * FROM password_resets WHERE email = ?",
            (email,)
        ).fetchone()

        if not row:
            return False, "Request an OTP first."

        if row["attempts"] >= MAX_OTP_ATTEMPTS:
            return False, "Too many attempts. Request a new OTP later."

        if now_utc() > parse_time(row["expires_at"]):
            return False, "OTP expired. Request a new one."

        if not check_password(otp.strip(), row["otp_hash"]):
            conn.execute(
                "UPDATE password_resets SET attempts = attempts + 1 WHERE email = ?",
                (email,)
            )
            return False, "Invalid OTP."

        conn.execute(
            "UPDATE password_resets SET verified_at = ? WHERE email = ?",
            (now_utc().isoformat(), email)
        )

    return True, "OTP verified successfully."


def reset_password(email, new_password):
    email = normalize_email(email)

    if not valid_password(new_password):
        return False, f"Password must be {MIN_PASSWORD_LENGTH}–72 characters."

    with connect_db() as conn:
        reset = conn.execute(
            "SELECT * FROM password_resets WHERE email = ?",
            (email,)
        ).fetchone()

        if not reset or not reset["verified_at"]:
            return False, "Verify your OTP first."

        if now_utc() > parse_time(reset["expires_at"]):
            return False, "OTP expired. Request a new one."

        user = conn.execute(
            "SELECT id FROM users WHERE email = ?",
            (email,)
        ).fetchone()

        if not user:
            return False, "Account not found."

        conn.execute(
            "UPDATE users SET password_hash = ? WHERE email = ?",
            (hash_password(new_password), email)
        )
        conn.execute(
            "DELETE FROM password_resets WHERE email = ?",
            (email,)
        )

    return True, "Password reset successful. Please sign in."


# ---------------- UI ----------------

st.markdown("""
<style>
.stApp {
    background: #0e1117;
}
.block-container {
    max-width: 650px;
    padding-top: 2rem;
}
.brand {
    text-align: center;
    padding: 12px 0 24px;
}
.brand h1 {
    font-size: 2.4rem;
    font-weight: 750;
    margin-bottom: 5px;
}
.brand p {
    color: #aab4c3;
    font-size: 1rem;
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="brand">
    <h1>🚢 FreightQuote AI</h1>
    <p>Maritime Intelligence Platform</p>
</div>
""", unsafe_allow_html=True)

mode = st.radio(
    "Authentication",
    ["Sign In", "Register", "Forgot Password"],
    horizontal=True,
    label_visibility="collapsed",
)

st.divider()

# ---------------- Sign In ----------------

if mode == "Sign In":
    st.subheader("Welcome back")

    with st.form("signin_form"):
        email = st.text_input("Email address").strip().lower()
        password = st.text_input("Password", type="password")
        submit = st.form_submit_button("Sign In", use_container_width=True)

    if submit:
        user = get_user_by_email(email) if email else None

        if user and check_password(password, user["password_hash"]):
            st.session_state["authenticated_email"] = email
            st.success(f"Signed in successfully. Welcome, {user['username']}!")
            st.info("Authentication is complete. FreightQuote features can be connected here.")
        else:
            st.error("Invalid email or password.")


# ---------------- Register ----------------

elif mode == "Register":
    st.subheader("Create an account")

    with st.form("register_form"):
        username = st.text_input("Username").strip()
        email = st.text_input("Email address").strip().lower()
        password = st.text_input("Password", type="password")
        confirm = st.text_input("Confirm password", type="password")
        submit = st.form_submit_button("Register", use_container_width=True)

    if submit:
        if not username or not email or not password:
            st.error("Please complete all fields.")
        elif "@" not in email or "." not in email.split("@")[-1]:
            st.error("Enter a valid email address.")
        elif not valid_password(password):
            st.error(f"Password must be {MIN_PASSWORD_LENGTH}–72 characters.")
        elif password != confirm:
            st.error("Passwords do not match.")
        else:
            try:
                with connect_db() as conn:
                    conn.execute("""
                        INSERT INTO users
                            (username, email, password_hash, role, created_at)
                        VALUES (?, ?, ?, 'Customer', ?)
                    """, (
                        username,
                        email,
                        hash_password(password),
                        now_utc().isoformat(),
                    ))
                st.success("Account created. Select Sign In to continue.")
            except sqlite3.IntegrityError:
                st.error("That username or email is already registered.")


# ---------------- Forgot Password ----------------

else:
    st.subheader("Reset your password")
    st.write("Request an email OTP, verify it, then choose a new password.")

    with st.form("request_otp_form"):
        email = st.text_input(
            "Registered email address",
            key="reset_email"
        ).strip().lower()
        request = st.form_submit_button("Send OTP", use_container_width=True)

    if request:
        if not email or "@" not in email:
            st.error("Enter a valid email address.")
        else:
            try:
                result = request_otp(email)
                if result:
                    st.session_state["otp_email"] = email
                    st.success(
                        "If this email belongs to an account, an OTP has been sent."
                    )
            except Exception:
                st.error(
                    "Email delivery failed. Check Colab Secrets and SMTP configuration."
                )

    reset_email = st.session_state.get("otp_email", "")

    if reset_email:
        st.caption(f"OTP requested for: {reset_email}")

        with st.form("verify_otp_form"):
            otp = st.text_input("6-digit email OTP", max_chars=6)
            verify = st.form_submit_button("Verify OTP", use_container_width=True)

        if verify:
            ok, message = verify_otp(reset_email, otp)
            if ok:
                st.session_state["otp_verified_email"] = reset_email
                st.success(message)
            else:
                st.error(message)

    verified_email = st.session_state.get("otp_verified_email", "")

    if verified_email:
        st.success("Email verified. Choose a new password.")

        with st.form("new_password_form"):
            new_password = st.text_input("New password", type="password")
            confirm_password = st.text_input("Confirm new password", type="password")
            change = st.form_submit_button(
                "Reset Password", use_container_width=True
            )

        if change:
            if new_password != confirm_password:
                st.error("Passwords do not match.")
            else:
                ok, message = reset_password(verified_email, new_password)
                if ok:
                    st.success(message)
                    st.session_state.pop("otp_email", None)
                    st.session_state.pop("otp_verified_email", None)
                else:
                    st.error(message)


Overwriting /content/FreightQuote_AI/app.py


In [ ]:
!streamlit run /content/FreightQuote_AI/app.py --server.port 8503 --server.address 0.0.0.0



2026-10-24 18:26:44.406 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://35.231.167.67:8503

  Stopping...
  Stopping...


In [ ]:
!streamlit run /content/FreightQuote_AI/app.py --server.port 8503 --server.address 0.0.0.0



2026-10-24 19:19:42.267 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://35.231.167.67:8503

  Stopping...
  Stopping...


In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
from google.colab import userdata

try:
    # Read token securely from Colab Secrets
    token = userdata.get("NGROK_AUTHTOKEN").strip()
    ngrok.set_auth_token(token)
except PyngrokNgrokError as e:
    if "already set" in str(e):
        print("Ngrok token already configured. Skipping setting step.")
    else:
        raise e

# Purane ngrok tunnels band karo
print("Disconnecting active tunnels...")
ngrok.kill()

# Port 8503 ke liye naya public URL banao
tunnel = ngrok.connect(8503, "http")
print("\n=============================================")
print("OPEN THIS URL:", tunnel.public_url)
print("=============================================")

Ngrok token already configured. Skipping setting step.
Disconnecting active tunnels...

OPEN THIS URL: https://reptile-unspoken-tuition.ngrok-free.dev


In [ ]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
from google.colab import userdata

try:
    # Securely read and set the authtoken
    token = userdata.get("NGROK_AUTHTOKEN").strip()
    ngrok.set_auth_token(token)
except PyngrokNgrokError as e:
    if "already set" in str(e):
        print("Ngrok token already configured. Skipping set step.")
    else:
        raise e

# Disconnect any previously active sessions
print("Disconnecting active tunnels...")
ngrok.kill()

# Create a new secure tunnel to port 8503
tunnel = ngrok.connect(8503, "http")
print("\n=============================================")
print("YOUR ACTUAL URL:", tunnel.public_url)
print("=============================================")

Ngrok token already configured. Skipping set step.
Disconnecting active tunnels...

YOUR ACTUAL URL: https://reptile-unspoken-tuition.ngrok-free.dev


In [ ]:
import subprocess
import time
import os
import shutil
from pathlib import Path

# Ensure we are in the correct directory
%cd /content/FreightQuote_AI

# Safely kill any existing streamlit runs on port 8503
!fuser -k 8503/tcp 2>/dev/null || true
!pkill -f "streamlit run"

time.sleep(2)

# Start Streamlit on port 8503 in the background
streamlit_bin = shutil.which("streamlit") or "/usr/local/bin/streamlit"
print("Starting Streamlit on port 8503 in background...")

with open("streamlit_server.log", "w") as log_file:
    process = subprocess.Popen(
        [streamlit_bin, "run", "app.py", "--server.port", "8503", "--server.address", "127.0.0.1"],
        stdout=log_file,
        stderr=log_file,
        text=True,
        preexec_fn=os.setsid
    )

time.sleep(5)

# Verify status
if process.poll() is None:
    print("SUCCESS: Streamlit is now actively running in the background on port 8503!")
else:
    print("FAILURE: Failed to start Streamlit. Checking logs:")
    if os.path.exists("streamlit_server.log"):
        with open("streamlit_server.log", "r") as f:
            print(f.read())

/content/FreightQuote_AI
Starting Streamlit on port 8503 in background...
SUCCESS: Streamlit is now actively running in the background on port 8503!


In [ ]:
from google.colab import userdata

for name in ["SMTP_EMAIL", "SMTP_APP_PASSWORD", "SMTP_EMAIL_PASSWORD"]:
    try:
        value = userdata.get(name)
        print(name, ":", "AVAILABLE" if value else "EMPTY")
    except Exception:
        print(name, ": NOT FOUND or Notebook access OFF")

SMTP_EMAIL : AVAILABLE
SMTP_APP_PASSWORD : AVAILABLE
SMTP_EMAIL_PASSWORD : NOT FOUND or Notebook access OFF


In [ ]:
import smtplib, ssl
from google.colab import userdata

email = userdata.get("SMTP_EMAIL")
password = userdata.get("SMTP_APP_PASSWORD").replace(" ", "")

with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
    server.ehlo()
    server.starttls(context=ssl.create_default_context())
    server.ehlo()
    server.login(email, password)

print("SMTP LOGIN SUCCESS")

SMTP LOGIN SUCCESS


In [ ]:
from email.message import EmailMessage
from google.colab import userdata
import smtplib, ssl

email = userdata.get("SMTP_EMAIL")
password = userdata.get("SMTP_APP_PASSWORD").replace(" ", "")

msg = EmailMessage()
msg["Subject"] = "FreightQuote AI - SMTP Test"
msg["From"] = email
msg["To"] = email
msg.set_content("SMTP test successful. Email delivery is working.")

with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
    server.ehlo()
    server.starttls(context=ssl.create_default_context())
    server.login(email, password)
    server.send_message(msg)

print("TEST EMAIL SENT")

TEST EMAIL SENT


In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
code = p.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(code):
    if "Email delivery failed" in line or "send_otp_email" in line:
        start = max(0, i - 5)
        end = min(len(code), i + 9)
        print(f"\n--- Lines {start+1}-{end} ---")
        for j in range(start, end):
            print(f"{j+1}: {code[j]}")


--- Lines 105-118 ---
105:         return value.strip() if value else None
106:     except Exception:
107:         return None
108: 
109: 
110: def send_otp_email(email, otp):
111:     sender = get_secret("SMTP_EMAIL")
112:     app_password = get_secret("SMTP_APP_PASSWORD")
113: 
114:     if not sender or not app_password:
115:         raise RuntimeError(
116:             "SMTP_EMAIL or SMTP_APP_PASSWORD is missing in Colab Secrets."
117:         )
118: 

--- Lines 171-184 ---
171:             return False
172: 
173:     otp = f"{secrets.randbelow(1_000_000):06d}"
174: 
175:     # Send first. Save the OTP hash only if email sending succeeds.
176:     send_otp_email(email, otp)
177: 
178:     with connect_db() as conn:
179:         conn.execute("""
180:             INSERT INTO password_resets
181:                 (email, otp_hash, expires_at, attempts, verified_at, last_sent_at)
182:             VALUES (?, ?, ?, 0, NULL, ?)
183:             ON CONFLICT(email) DO UPDATE SET
184:        

In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
lines = p.read_text(encoding="utf-8").splitlines()

for i in range(105, 145):
    if i < len(lines):
        print(f"{i+1}: {lines[i]}")

106:     except Exception:
107:         return None
108: 
109: 
110: def send_otp_email(email, otp):
111:     sender = get_secret("SMTP_EMAIL")
112:     app_password = get_secret("SMTP_APP_PASSWORD")
113: 
114:     if not sender or not app_password:
115:         raise RuntimeError(
116:             "SMTP_EMAIL or SMTP_APP_PASSWORD is missing in Colab Secrets."
117:         )
118: 
119:     message = EmailMessage()
120:     message["Subject"] = "FreightQuote AI Password Reset OTP"
121:     message["From"] = sender
122:     message["To"] = email
123:     message.set_content(
124:         f"""Your FreightQuote AI password reset code is: {otp}
125: 
126: This code expires in {OTP_MINUTES} minutes.
127: If you did not request this code, ignore this email.
128: """
129:     )
130: 
131:     context = ssl.create_default_context()
132: 
133:     with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
134:         server.ehlo()
135:         server.starttls(context=context)
136:         

In [ ]:
from pathlib import Path

p = Path("/content/FreightQuote_AI/app.py")
lines = p.read_text(encoding="utf-8").splitlines()

for i in range(105, 145):
    if i < len(lines):
        print(f"{i+1}: {lines[i]}")

106:     except Exception:
107:         return None
108: 
109: 
110: def send_otp_email(email, otp):
111:     sender = get_secret("SMTP_EMAIL")
112:     app_password = get_secret("SMTP_APP_PASSWORD")
113: 
114:     if not sender or not app_password:
115:         raise RuntimeError(
116:             "SMTP_EMAIL or SMTP_APP_PASSWORD is missing in Colab Secrets."
117:         )
118: 
119:     message = EmailMessage()
120:     message["Subject"] = "FreightQuote AI Password Reset OTP"
121:     message["From"] = sender
122:     message["To"] = email
123:     message.set_content(
124:         f"""Your FreightQuote AI password reset code is: {otp}
125: 
126: This code expires in {OTP_MINUTES} minutes.
127: If you did not request this code, ignore this email.
128: """
129:     )
130: 
131:     context = ssl.create_default_context()
132: 
133:     with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
134:         server.ehlo()
135:         server.starttls(context=context)
136:         

In [ ]:
from google.colab import userdata
import smtplib
import ssl
from email.message import EmailMessage

try:
    sender = userdata.get("SMTP_EMAIL")
    password = userdata.get("SMTP_APP_PASSWORD")

    print("SMTP_EMAIL available:", bool(sender))
    print("SMTP_APP_PASSWORD available:", bool(password))

    if not sender or not password:
        raise RuntimeError("Required Colab Secret is unavailable.")

    msg = EmailMessage()
    msg["Subject"] = "FreightQuote AI OTP Diagnostic"
    msg["From"] = sender.strip()
    msg["To"] = sender.strip()
    msg.set_content("Diagnostic email test successful.")

    with smtplib.SMTP("smtp.gmail.com", 587, timeout=20) as server:
        server.ehlo()
        server.starttls(context=ssl.create_default_context())
        server.ehlo()
        server.login(sender.strip(), password.replace(" ", "").strip())
        server.send_message(msg)

    print("PASS: Diagnostic email sent.")

except Exception as e:
    print("FAIL:", type(e).__name__, str(e))

SMTP_EMAIL available: True
SMTP_APP_PASSWORD available: True
PASS: Diagnostic email sent.


In [ ]:
!streamlit run /content/FreightQuote_AI/app.py --server.port 8503 --server.address 0.0.0.0



2026-10-24 20:14:25.293 Port 8503 is not available


In [ ]:
!ps -ef | grep '[s]treamlit run'

root       87962   62030  0 19:30 ?        00:00:16 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8503 --server.address 127.0.0.1


In [ ]:
import os, signal

os.kill(87962, signal.SIGTERM)
print("Streamlit process stop signal sent.")

Streamlit process stop signal sent.


In [ ]:
!fuser -n tcp 8503

In [ ]:
!streamlit run /content/FreightQuote_AI/app.py --server.port 8503 --server.address 0.0.0.0

2026-10-24 20:17:45.798 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://35.231.167.67:8503



  Stopping...


In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))

tunnel = ngrok.connect(8503, "http")
print("YOUR FREIGHTQUOTE AI URL:", tunnel.public_url)

PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'